# Transformer Architecture Demo
Verification of sequence modeling logic using LSTM, Attention, BERT, and GPT.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import urllib.request
import math
import csv
import io
import re

%matplotlib inline
plt.style.use('dark_background')
torch.manual_seed(42)

# --- SVD-Character Foundation ---
s = "ROMEO AND JULIET. "
chs = sorted(list(set(s)))
c2i = {c: i for i, c in enumerate(chs)}; i2c = {i: c for i, c in enumerate(chs)}; vs_c = len(chs)

# --- Mini-GloVe Foundation ---
v_words = ["<PAD>", "<MASK>", "a", "rose", "by", "any", "other", "name", "would", "smell", "as", "sweet", "."]
w2i = {w: i for i, w in enumerate(v_words)}; i2w = {i: w for i, w in enumerate(v_words)}; vs_w = len(v_words)

EMBED_DIM = 32
WV = torch.randn(vs_w, EMBED_DIM) * 0.1
WV[w2i["rose"]] += 1.0; WV[w2i["name"]] += 0.8
WV[w2i["smell"]] += 0.5; WV[w2i["sweet"]] += 0.8

def get_word_em(word): return WV[w2i[word]]



## LSTM Forecasting (Airline Dataset)


In [ ]:
from modules.lstm import LSTM

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = [float(row[1]) for row in list(csv.reader(io.StringIO(urllib.request.urlopen(url).read().decode('utf-8'))))[1:]]
mi, ma = min(data), max(data)
norm = torch.tensor([(x - mi) / (ma - mi) for x in data]).unsqueeze(-1)

X, Y = [], []
for i in range(len(norm)-12):
    X.append(norm[i:i+12]); Y.append(norm[i+12])
X, Y = torch.stack(X), torch.stack(Y)

model = LSTM(1, 64); head = nn.Linear(64, 1)
opt = optim.Adam(list(model.parameters()) + list(head.parameters()), lr=0.01)

for _ in range(300):
    opt.zero_grad()
    loss = nn.MSELoss()(head(model(X[:110])[0][:,-1,:]), Y[:110])
    loss.backward(); opt.step()

with torch.no_grad():
    res = head(model(X[110:])[0][:,-1,:]).squeeze().tolist()
    true = Y[110:].squeeze().tolist()

plt.figure(figsize=(10, 4))
plt.plot(range(110, 110+len(true)), [t*(ma-mi)+mi for t in true], color='white', alpha=0.3, lw=3, label='Actual')
plt.plot(range(110, 110+len(res)), [r*(ma-mi)+mi for r in res], color='cyan', ls='--', label='LSTM')
plt.title("Airline Passenger Forecast")
plt.legend(); plt.show()



## Attention Weight Mapping


In [ ]:
from modules.attention import MultiHeadAttention

mha = MultiHeadAttention(d_model=32, num_heads=2)
demo_words = ["rose", "smell", "sweet", "by", "other", "name"]
x = torch.stack([get_word_em(w) for w in demo_words]).unsqueeze(0)

_, w = mha(x, x, x)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for h in range(2):
    axes[h].imshow(w[0, h].detach().tolist(), cmap='magma' if h==0 else 'inferno')
    axes[h].set_title(f"Head {h+1}")
    axes[h].set_xticks(range(6)); axes[h].set_xticklabels(demo_words, rotation=45)
    axes[h].set_yticks(range(6)); axes[h].set_yticklabels(demo_words)
plt.tight_layout(); plt.show()



## BERT Masked Language Modeling


In [ ]:
from modules.bert import BERT

bert = BERT(vocab_size=vs_w, d_model=32, num_heads=4, num_layers=4)
with torch.no_grad():
    bert.encoder.token_embedding.weight.copy_(WV)

opt = optim.Adam(bert.parameters(), lr=0.001)

quote = "a rose by any other name would smell as sweet"
tgt = torch.tensor([[w2i[w] for w in quote.split()]])
msk = tgt.clone(); msk[0, 5] = w2i["<MASK>"]

for _ in range(500):
    opt.zero_grad()
    loss = nn.CrossEntropyLoss()(bert(msk)[0].view(-1, vs_w), tgt.view(-1))
    loss.backward(); opt.step()

res = torch.argmax(bert(msk)[0][0, 5, :]).item()
print(f"BERT prediction for '[MASK]': {i2w[res]}")



## GPT Autoregressive Spelling


In [ ]:
from modules.gpt import GPT

X_gpt = torch.tensor([[c2i[c] for c in s[:-1]]]).repeat(10, 1)
Y_gpt = torch.tensor([[c2i[c] for c in s[1:]]]).repeat(10, 1)

gpt = GPT(vocab_size=vs_c, d_model=64, num_heads=4, num_layers=4)
opt = optim.Adam(gpt.parameters(), lr=0.001)

for _ in range(1200):
    opt.zero_grad()
    loss = nn.CrossEntropyLoss()(gpt(X_gpt).view(-1, vs_c), Y_gpt.view(-1))
    loss.backward(); opt.step()

ctx = torch.tensor([[c2i["R"]]])
for _ in range(16):
    nxt = torch.argmax(gpt(ctx)[0, -1, :]).item()
    ctx = torch.cat([ctx, torch.tensor([[nxt]])], dim=1)

print(f"GPT Spelling: {''.join([i2c[i] for i in ctx[0].tolist()])}")

